In [ ]:
from src.agent.prompts import (
    get_current_date,
    query_writer_instructions,
    web_searcher_instructions,
    reflection_instructions,
    answer_instructions,
)
from src.agent.state import (
    OverallState,
    QueryGenerationState,
    ReflectionState,
    WebSearchState,
)

In [ ]:

import dotenv
from rich import print as rprint
from src.agent.llm.llm import OpenAICompatibleLLM
from src.agent.base_agent import Agent
dotenv.load_dotenv(override=True)

model = 'MiniMax-M3'
deep_research_topic = '现在企业级的 AI Agent 落地方案深度调研'
initial_search_query_count = 2

## Agent class 说明

- set_step_prompt() ： 设置提示词

- step():
  - 使用 以上设置好的提示词，调用 LLM
  - 对结果进行后期处理（直接返回/json/解析异常等）
- __call__ ：llm 调用


In [ ]:

agent = Agent(model_id=model)
print('=============== prompt before format ===============')
rprint(query_writer_instructions)


agent.set_step_prompt(query_writer_instructions)
print('=============== prompt after format ===============')
prompt_formatted=agent.prompt_format(
    prompt=query_writer_instructions,
    current_date=get_current_date(),
    research_topic=deep_research_topic,
    number_queries=initial_search_query_count)
rprint(prompt_formatted)


response = agent.step(
  current_date=get_current_date(),
  research_topic=deep_research_topic,
  number_queries=initial_search_query_count,
)
rprint( 'response is str :',isinstance(response,str))
rprint(response)

## JsonAgent

- 指定一个 json 对象格式化的类
- 格式化输出
  - 去除 LLM 的 <think> 等标签
  - 正则方式 ： 只获取 ```{{ ... }} ``` 中的内容

### generate_search

In [ ]:

from src.agent.base_agent import JsonAgent
from src.agent.tools_and_schemas import SearchQueryList, Reflection

agent = JsonAgent(model_id=model, keys=SearchQueryList)

init_prompt = query_writer_instructions
print('===============原始提示词， prompt before format ===============')
rprint(init_prompt)


agent.set_step_prompt(init_prompt)
print('===============提示词格式化， prompt after format ===============')
prompt_formatted=agent.prompt_format(
    prompt=init_prompt,
    current_date=get_current_date(),
    research_topic=deep_research_topic,
    number_queries=initial_search_query_count)
rprint(prompt_formatted)
print('=============== 调用 LLM，并 做 后期 post process 并返回结果 ===============')

result:SearchQueryList = agent.step(
    current_date=get_current_date(),
    research_topic=deep_research_topic,
    number_queries=initial_search_query_count,
)
rprint( 'result is SearchQueryList :',isinstance(result,SearchQueryList), type(result))

rprint(result)

### send_to_web_search

In [ ]:
from agent.graph import WEB_SEARCH_NODE
from langgraph.types import Send
state = {
    'search_query':result.query
}
sends= [
    Send(WEB_SEARCH_NODE, {"search_query": search_query, "id": int(idx)})
    for idx, search_query in enumerate(state["search_query"])
]

rprint(sends)

### web_search



In [ ]:
from src.agent.base_agent import WebSearchAgent

dotenv.load_dotenv(override=True)
for send in sends:
    arg = send.arg
    rprint(arg)
    web_searcher = WebSearchAgent()

    # 执行搜索
    response = web_searcher.step(prompt=arg["search_query"],
                                 count=3)
    rprint(response)

# MCP Agent

[
    Send(node='web_search', arg={'search_query': '2025 2026 企业级 AI Agent 框架平台对比 LangChain AutoGen CrewAI
Salesforce Agentforce Microsoft Copilot Studio 落地趋势', 'id': 0}),
    Send(node='web_search', arg={'search_query': '企业 AI Agent 落地案例 架构设计 部署实践 挑战与经验 金融 制造
客服场景', 'id': 1})
]

In [ ]:
from dashscope import Application
from agent.base_agent import get_web_search_rate_limiter, MCPAgent
import os

search_list = [
    '2025 2026 企业级 AI Agent 框架平台对比 LangChain AutoGen CrewAI Salesforce Agentforce Microsoft Copilot Studio 落地趋势',
    '企业 AI Agent 落地案例 架构设计 部署实践 挑战与经验 金融 制造 客服场景'
]

dotenv.load_dotenv(override=True)

api_key = os.getenv("MCP_APP_TOKEN")
app_id = os.getenv("MCP_APP_ID")
for search in search_list:
    step_prompt = search

    rate_limiter = get_web_search_rate_limiter()
    wait_time=rate_limiter.acquire()
    if wait_time > 0:
        print(f"速率限制等待: {wait_time:.3f}秒")

    response = Application.call(
        api_key=api_key,
        app_id=app_id,
        prompt = step_prompt,
        biz_params={'count':2}
    )

    ## post_process 结果解析 && 异常处理
    response = WebSearchAgent().post_process(response)
    rprint(response)